# 2. Learning spatial relationships with MISTy

While Moran's R provides a sound summary of spatial clustering, it is limited to two variables at a time and is thus not fit for complex, or non-linear, spatial relationships between variables. 

Here, we show how to use LIANA's implementation of [MISTy](https://github.com/saezlab/mistyR), a framework presented in [Tanevski et al., 2022](https://genomebiology.biomedcentral.com/articles/10.1186/s13059-022-02663-5).

MISTy is a tool that helps us better understand how different features, such as genes or cell types, interact with each other in space. MISTy does so by learning both **intra-** and **extra-view** relationships - i.e. those that occur within the same cell/spot and between neighbouring cells/spots. On Visium, a spot contains several cells, so "intra" means *within a spot*, not intracellular. A major advantage of MISTy is its flexibility. It can model different perspectives, or "views," each describing a different way markers are related to each other. Each of these views can describe a different spatial context, i.e. define a relationship among the observed expressions of the markers, such as relationships within a spot or paracrine regulation.

MISTy has only one fixed view - i.e. the intraview, which contains the target (dependent) variables. The other views we refer to as extra views, and they contain the independent variables used to predict the intra view. MISTy can fit any number of extra views, and each extra view can contain any number of variables. The extra views can thus simultaneously learn the dependencies of target variables across different modalities, such as cell type proportions, pathways, or genes, etc.

MISTy treats each view as a potential source of variation in the measurements of the target variables in the intra view. MISTy further analyzes each view to determine how it contributes to the overall expression or abundance of each target variable. It explains this contribution by identifying the interactions between measurements that led to the observed results.



### Import packages

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)  # silence squidpy deprecation warnings

import scanpy as sc
import squidpy as sq
from matplotlib import pyplot as plt
import decoupler as dc
import plotnine as p9
import liana as li

### Import functions needed to create MISTy objects.

In [ ]:
from liana.method import MistyData, genericMistyData

### Import Pre-defined Single view models

In [ ]:
from liana.method.sp import RandomForestModel, LinearModel, RobustLinearModel

### Load and Normalize Data

We use the same Visium slide from [Kuppe et al., 2022](https://www.nature.com/articles/s41586-022-05060-x).

In [ ]:
import os
# li.ds caches downloads under ../../../datasets/Hands_on_2_LIANA_MistY/anndata; a provided Visium_19_CK297.h5ad there is used as-is
sc.settings.datasetdir = os.path.join(os.environ.get("MOPITAS_DATA", "../../../datasets"), "Hands_on_2_LIANA_MistY")
adata = li.ds.kuppe_2022()

In [ ]:
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

##### Extract Cell type Composition
This slide comes with estimated cell type proportions using cell2location; See [Kuppe et al., 2022](https://www.nature.com/articles/s41586-022-05060-x). Let's extract from .obsm them to an independent AnnData object.

In [ ]:
# Rename to more informative names
full_names = {'Adipo': 'Adipocytes',
              'CM': 'Cardiomyocytes',
              'Endo': 'Endothelial',
              'Fib': 'Fibroblasts',
              'PC': 'Pericytes',
              'prolif': 'Proliferating',
              'vSMCs': 'Vascular_SMCs',
              }
# but only for the ones that are in the data
adata.obsm['compositions'].columns = [full_names.get(c, c) for c in adata.obsm['compositions'].columns]

In [ ]:
comps = li.pp.obsm_to_adata(adata, 'compositions')

In [ ]:
comps.var

In [ ]:
# check key cell types
sq.pl.spatial_scatter(comps,
                       color=['Vascular_SMCs','Cardiomyocytes',
                              'Endothelial', 'Fibroblasts'],
                       size=1.3, ncols=2, img_alpha=0
                       )
plt.show()

## Formatting & Running MISTy


The implementation of MISTy in liana+ relies on [MuData](https://github.com/scverse/mudata) objects [(Bredikhin et al., 2022)](https://genomebiology.biomedcentral.com/articles/10.1186/s13059-021-02577-8) and extends them to a very simple child class we call "MistyData". 
To make it easier to use, we provide functions to construct "MistyData" objects that transform the data into a format that MISTy can use.

Briefly, a "MistyData" object is just a MuData object with **intra** as one of the modalities - this is the view in which the (**target**) variables explained by all other views are stored. 
MISTy is flexible to any other view that is appended, provided it also contains a spatial neighbors graph.


Let's use `genericMistyData` to construct it with the cell type proportions as the intra view.
It additionally builds a 'juxta' view for the spots that are neighbors of each other, and a 'para' view for all surrounding spots within a certain radius, or bandwidth.

In [ ]:
misty = genericMistyData(intra=comps, cutoff=0.05, bandwidth=200, n_neighs=6)

In [ ]:
misty

## Learn Relationships with MISTy


In [ ]:

misty(model=LinearModel, k_cv=10, seed=1337, verbose = True, bypass_intra=True)

#misty(model=RandomForestModel, n_jobs=-1, verbose = True) # you can also use RandomForestModel but it takes longer time

Specifically, we will use the `LinearModel` to fit an linear model for each target in the intra view, using the juxta and para views as predictors. It is a bit more simplistic but much faster and more interpretable.
You can also use `RandomForestModel` but it takes longer time.

We also set `bypass_intra=True`, i.e. we do not predict each cell type from the other cell types *in the same spot*. Proportions sum to 1, so the other proportions in the same spot would predict the target almost exactly (R² ≈ 1 for all cell types but adipocytes) and leave nothing for the spatial views to explain.

MISTy returns two DataFrames:
* `target_metrics` - the metrics that describe the target variables from the intra view, including R-squared across different views as well as the estimated contributions to the predictive performance of each view per target.
* `interactions` - feature importances per view

if `inplace` is true (Default), these are appended to the MuData object.


Let's look at the target metrics. Because we bypassed the intra view, `intra_R2` is 0 and `gain_R2` equals `multi_R2`, the variance of each cell type's proportion explained by the proportions of the *other* cell types in the surrounding spots.

In [ ]:
misty.uns['target_metrics']

`multi_R2` shows how much of each target is predictable from the extra views. Be careful with compositions: since proportions sum to 1, the other cell types in the neighbouring spots also tell us how much of the target is there, so a high R² largely reflects how smooth the target is in space.

(Without `bypass_intra`, `gain_R2` would be the improvement of the extra views over the intra view alone.)

In [ ]:
li.pl.misty_target_metrics(misty, stat='multi_R2', return_fig=True)

We can also check the contribution of each view per target. These are the meta-model's weights, clipped to be non-negative and normalised to sum to 1, not fractions of variance explained. The 6 juxta neighbours are also part of the para neighbourhood, so the two views are strongly collinear and the split between them can be unstable.

In [ ]:
li.pl.misty_contributions(misty, return_fig=True)

## Check co-localised cell types

Each tile shows the importance of a predictor (in the para view) for a target, here a t-value: its sign gives the direction of the relationship and its size the strength. Because proportions sum to 1, when one cell type goes up the others must go down, so negative values are largely forced by closure. Only strong positive values suggest co-localisation.

In [ ]:
(
    li.pl.misty_interactions(misty, view='para', return_fig=True, figure_size=(7,5)) + 
    # diverging scale centred at 0: t-values are signed
    p9.scale_fill_gradient2(low='#3b4cc0', mid='#f7f7f7', high='#b40426', midpoint=0)
)

### Add pathway activities as extra view

We can also use estimated pathway activities as extra views to make the data a bit more interpretable.
We will use [decoupler-py](https://academic.oup.com/bioinformaticsadvances/article/2/1/vbac016/6544613) with pathways genesets from [PROGENy](https://www.nature.com/articles/s41467-017-02391-6). See [this tutorial](https://decoupler-py.readthedocs.io/en/latest/notebooks/spatial.html) for details.

In [ ]:
# obtain genesets
progeny = dc.op.progeny(organism='human', top=100)

In [ ]:
# use univariate linear model to estimate activity
dc.mt.ulm(
    adata,
    net=progeny,
    tmin=5,
    verbose=True,
    bsize = 256,
    raw=False
)

In [ ]:
# extract progeny activities as an AnnData object
acts_progeny = li.pp.obsm_to_adata(adata, 'score_ulm')

In [ ]:
# Check how the pathway activities look like
sq.pl.spatial_scatter(acts_progeny, color=['Hypoxia', 'JAK-STAT'], cmap='RdBu_r', size=1.3)
plt.show()

In this case, we will use cell type compositions per spot as the intra view, and we will use the PROGENy pathway activities to define the juxta and para views:

In [ ]:
misty = genericMistyData(intra=comps, extra=acts_progeny, cutoff=0.05, bandwidth=200, n_neighs=6)

In [ ]:
misty

#### Run MISTy with pathway views

As before, we bypass the intra view (`bypass_intra=True`), so the R² reflects only what the pathway views explain.

In [ ]:
misty(model=LinearModel, k_cv=10, seed=1337, bypass_intra=True, verbose = True)

Let's check the joined R-squared for views:

In [ ]:
li.pl.misty_target_metrics(misty, stat='multi_R2', return_fig=True)

We can now also see which specific variables explain each target best:

In [ ]:
# this information is stored here:
misty.uns['interactions'].head()

and their contributions per target. Note how unstable the juxta/para split is: some targets are assigned entirely to one of the two views.

In [ ]:
li.pl.misty_contributions(misty, return_fig=True)

Since this is a linear model, the coefficients would not be directly comparable (as are importances in a Random Forest). Thus, we use the coefficients' t-values, as calculated by Ordinary Least Squares, which are signed and directly comparable.

Let's explore the t-values for each target-prediction interaction:

In [ ]:
(
    li.pl.misty_interactions(misty, view='juxta', return_fig=True, figure_size=(7,5)) + 
    # diverging scale centred at 0: t-values are signed
    p9.scale_fill_gradient2(low='#3b4cc0', mid='#f7f7f7', high='#b40426', midpoint=0)
)

<div class="alert alert-info">
    
<h4> Feature importances </h4>

Here we use `LinearModel`, so the importances are the t-values of the model coefficients. They are signed, so we plot them on a diverging colour scale centred at 0. With `RandomForestModel`, the importances are the random forest's feature importances, i.e. the (normalised) decrease in squared error from splits on each feature. They are non-negative, so use a sequential colour scale.


</div>  

## Build Custom Misty Views

As we previously mentioned, one can build any view structure that they deem relevant for their data.
So, let's explore how to build custom views.
Here, we will just use two distinct prior knowledge sources to check which one achieves better predictive performance.

So, let's also estimate Transcription Factor activities with decoupler:

In [ ]:
# get TF prior knowledge
net = dc.op.collectri(organism='human', remove_complexes=True, license='academic', verbose=False)

In [ ]:
# Estimate activities
dc.mt.ulm(
    data=adata,
    net=net,
    bsize=128,  
    tmin = 50,
    verbose=True,
    raw=False
)

In [ ]:
# extract activities
acts_tfs = li.pp.obsm_to_adata(adata, 'score_ulm')

In [ ]:
# Calculate spatial neighbors
li.pp.spatial_neighbors(acts_tfs, cutoff=0.1, bandwidth=200, set_diag=True) # you can also set set_diag=False

Visualize the weights for a specific spot:

In [ ]:
# connectivity weights between the first spot and all other spots
acts_tfs.obs['connectivity'] = acts_tfs.obsp['spatial_connectivities'][:, 0].toarray().ravel()
sq.pl.spatial_scatter(acts_tfs, color='connectivity', size=1.3, img_alpha=0)
plt.show()

In [ ]:
# transfer spatial information to progeny activities
# NOTE: spatial connectivities can differ between views, but in this case we will use the same
acts_progeny.obsm['spatial'] = acts_tfs.obsm['spatial']
acts_progeny.obsp['spatial_connectivities'] = acts_tfs.obsp['spatial_connectivities']

Build an object with custom views:

In [ ]:
misty = MistyData(data={"intra": comps, "TFs": acts_tfs, "Pathways": acts_progeny})

In [ ]:
misty

Run Misty as before:

In [ ]:
misty(model=LinearModel, k_cv=5, seed=1337, bypass_intra=True, verbose = True)

We can see that Cardiomyocytes, Fibroblasts and Vascular SMCs are relatively well explained by TFs & Pathways.

In [ ]:
li.pl.misty_target_metrics(misty, stat='multi_R2')

The TF view receives most of the weight for every target (≈ 58–93 %). This is not a fair comparison of the two knowledge sources, though: the TF view has 170 predictors versus 14 for pathways.

In [ ]:
li.pl.misty_contributions(misty, return_fig=True)

Plot cell type x transcription factor interactions:

In [ ]:
(
    # key=abs: top 20 by |t|, so negative values are kept
    li.pl.misty_interactions(misty, view='TFs', top_n=20, key=abs) + 
    p9.labs(x='Transcription Factor', y='Cell type') +
    p9.theme_bw(base_size=14) +
    p9.theme(axis_text_x=p9.element_text(rotation=90, size=13)) +
    # diverging scale centred at 0: t-values are signed
    p9.scale_fill_gradient2(low='#3b4cc0', mid='#f7f7f7', high='#b40426', midpoint=0) +
    p9.theme(figure_size=(8, 5)) 
)


---

<div style="border-left:5px solid #2e7d32; background:#f1f8e9; padding:10px 14px; border-radius:4px; margin:12px 0;">

**🧪 Exercise — Where is a ligand–receptor interaction active?**

This exercise combines the previous notebook with this one. Use the local LR scores from
`li.mt.bivariate` as MISTy **targets**, and ask: *which cell types — in the same spot, or
in the surrounding tissue — explain where each interaction is active?*

1. Compute spatial neighbours on `adata` (`bandwidth=150`) and run `li.mt.bivariate`
   (`local_name='cosine'`, `global_name='morans'`, `nz_prop=0.05`).
2. Keep the 20 most spatially variable interactions (highest `morans`) as the intra view.
3. Build two views from the cell-type compositions `comps`:
   - `local`: the composition of the **same spot** — give it an identity connectivity,
     `scipy.sparse.identity(comps.n_obs, format='csr')`, in `.obsp['spatial_connectivities']`;
   - `nearby`: the composition of the **neighbourhood** — `li.pp.spatial_neighbors` with
     `bandwidth=200`, `cutoff=0.05` and `set_diag=False`.
4. Fit `MistyData(data={'intra': ..., 'local': ..., 'nearby': ...})` with a
   `RandomForestModel` and `bypass_intra=True`, so that only cell types (not other LR
   pairs) are used as predictors. (A MuData warning about non-unique `var_names` is
   expected: `local` and `nearby` share the cell-type names.)

Steps 1–2 are pre-filled in the cell below. Then look at which interactions are well explained, which view explains them, and which
cell type is behind the best-explained ones.
</div>

In [ ]:
# your turn
from scipy import sparse

# 1. local LR scores (pre-filled, as in the bivariate notebook)
li.pp.spatial_neighbors(adata, bandwidth=150, cutoff=0.1, kernel='gaussian', set_diag=True)
lrdata = li.mt.bivariate(adata, resource_name='consensus', local_name='cosine',
                         global_name='morans', nz_prop=0.05, use_raw=False)

# 2. targets: the 20 most spatially variable interactions (pre-filled)
top_lrs = lrdata.var.sort_values('morans', ascending=False).head(20).index
lr_targets = lrdata[:, top_lrs].copy()

# 3. build the `local` and `nearby` views from `comps`


# 4. fit MistyData with a RandomForestModel and bypass_intra=True


<details>
<summary><b>Solution</b></summary>

```python
from scipy import sparse

# 1. local LR scores (as in the bivariate notebook)
li.pp.spatial_neighbors(adata, bandwidth=150, cutoff=0.1, kernel='gaussian', set_diag=True)
lrdata = li.mt.bivariate(adata, resource_name='consensus', local_name='cosine',
                         global_name='morans', nz_prop=0.05, use_raw=False)

# 2. targets: the 20 most spatially variable interactions
top_lrs = lrdata.var.sort_values('morans', ascending=False).head(20).index
lr_targets = lrdata[:, top_lrs].copy()

# 3. predictors: cell-type compositions in the same spot and in the neighbourhood
local = comps.copy()
local.obsp['spatial_connectivities'] = sparse.identity(comps.n_obs, format='csr')
nearby = comps.copy()
li.pp.spatial_neighbors(nearby, bandwidth=200, cutoff=0.05, set_diag=False)

# 4. fit; bypass_intra so that LR pairs are not used to predict each other
misty = MistyData(data={'intra': lr_targets, 'local': local, 'nearby': nearby})
misty(model=RandomForestModel, n_jobs=-1, seed=1337, bypass_intra=True, verbose=True)
```

**Which interactions are explained?** Because `bypass_intra=True`, `intra_R2` is 0 and
`multi_R2` is the variance explained by cell types alone.

```python
li.pl.misty_target_metrics(misty, stat='multi_R2')
```

All 20 interactions are well explained (R² 0.81–0.92): where an interaction is active is
largely determined by which cell types are there.

**Here or nearby?** The contributions show which view carries that information.

```python
li.pl.misty_contributions(misty)
```

The `nearby` view dominates (≈ 90–100 % for every target). Part of this is biology — signalling
depends on the surrounding tissue — but part of it is by construction: the local bivariate
score is itself computed over a neighbourhood (`bandwidth=150`), so a neighbourhood-level
composition matches it best. Keep this in mind before reading "nearby" as "paracrine".

**Which cell types?** Importances from a random forest are non-negative, so use a
sequential colour scale.

```python
(li.pl.misty_interactions(misty, view='nearby', top_n=20, return_fig=True)
 + p9.scale_fill_gradient(low='#fff5eb', high='#b30000')
 + p9.theme(figure_size=(8, 5)))
```

Two groups stand out. The collagen–integrin interactions (`COL1A1^ITGA11_ITGB1`,
`COL1A2^…`, `COL3A1^…`, …) are explained by **fibroblasts** — collagen deposition by
fibroblasts through integrin α11β1 is a hallmark of the fibrotic, ischemic zone. Interactions
such as `S100A1^RYR2`, `LPL^VLDLR` and `VTN^ITGAV_ITGB1` are explained by
**cardiomyocytes**, i.e. they mark the remaining viable myocardium.

**Check one on the tissue.** Put the interaction next to its top predictor, in the same
spot and averaged over the neighbourhood. MISTy applies the `nearby` weights internally,
so we compute that neighbourhood average ourselves to see it:

```python
import numpy as np

W = nearby.obsp['spatial_connectivities']
comps_nearby = comps.copy()
comps_nearby.X = np.asarray(W @ comps.X) / np.asarray(W.sum(axis=1))

sq.pl.spatial_scatter(lr_targets, color='COL1A1^ITGA11_ITGB1', cmap='magma', size=1.3)
sq.pl.spatial_scatter(comps, color='Fibroblasts', cmap='viridis', size=1.3,
                      title='Fibroblasts (same spot)')
sq.pl.spatial_scatter(comps_nearby, color='Fibroblasts', cmap='viridis', size=1.3,
                      title='Fibroblasts (neighbourhood average)')
plt.show()
```

The interaction map follows the fibroblast-rich regions, and it matches the smooth
neighbourhood average more closely than the noisy per-spot proportion (correlation ≈ 0.6
vs ≈ 0.45) — which is what the contributions plot said.

**Caveats.** Random-forest importances have no sign: "explained by cardiomyocytes" can also
mean that cardiomyocytes are *absent* (e.g. for `S100A9^CD68`), so check the direction on the
map. And the compositions and the LR scores come from the same transcriptome (COL1A1 is
itself a fibroblast marker), so MISTy shows *where* an interaction is active, not *who*
signals.

</details>

<div style="border-left:5px solid #2e7d32; background:#f1f8e9; padding:10px 14px; border-radius:4px; margin:12px 0;">

**🧪 Optional — Does the relationship depend on the niche?**

`adata.obs['celltype_niche']` assigns each spot to one of nine cell-type niches ([Kuppe et al., 2022](https://www.nature.com/articles/s41586-022-05060-x)); `lr_targets` inherits this column. Refit the exercise model with `maskby='celltype_niche'`, so that MISTy fits a separate model within each niche. Compare the R² across niches, and check whether the top `nearby` predictor of each interaction changes between niches.
</div>

In [ ]:
# your turn


<details>
<summary><b>Solution</b></summary>

```python
import numpy as np

misty_niche = MistyData(data={'intra': lr_targets, 'local': local, 'nearby': nearby})
misty_niche(model=RandomForestModel, n_jobs=-1, seed=1337, bypass_intra=True,
            maskby='celltype_niche', verbose=True)
```

The results now have one row per target *and niche* (column `intra_group`). ctniche_2 and ctniche_8 have only ~70 spots, too few for stable fits (ctniche_8 has R² ≈ 0 for almost every target, ctniche_2 for a quarter of them), so we leave them out. Passing `aggregate_fn` draws one boxplot per target across the niches:

```python
small = ['ctniche_2', 'ctniche_8']
li.pl.misty_target_metrics(misty_niche, stat='multi_R2', aggregate_fn=np.mean,
                           filter_fn=lambda x: x['intra_group'] not in small,
                           figure_size=(8, 5))
```

Within a niche the R² drops (per-niche medians ≈ 0.5–0.7, versus 0.81–0.92 on the whole slide): part of the whole-slide fit came from differences *between* regions.

What do the niches look like, and which cell type is the top `nearby` predictor in each?

```python
comps.to_df().groupby(adata.obs['celltype_niche']).mean().round(2)
```

```python
imp = misty_niche.uns['interactions'].query("view == 'nearby'")
top = imp.sort_values('importances', ascending=False).groupby(['target', 'intra_group']).head(1)
top.pivot(index='target', columns='intra_group', values='predictor').drop(columns=small)
```

ctniche_4 and ctniche_5 are dominated by fibroblasts (≈ 0.55) and myeloid cells (≈ 0.2) and contain almost no cardiomyocytes, whereas ctniche_1, 3 and 7 are mixed. In most niches fibroblasts remain the top predictor of the collagen–integrin interactions. In ctniche_5, where fibroblasts are everywhere, myeloid cells take over for COL1A1, COL1A2, COL3A1 and COL5A1. Likewise, where cardiomyocytes are nearly absent (ctniche_4, 5), several cardiomyocyte-linked interactions (e.g. `BSG^SLC16A1`, `MFGE8^ITGAV`) are explained by other cell types instead. This reflects which cell types vary within that niche, not rewired signalling. Only the targets are masked: the `nearby` view still averages over spots from adjacent niches.

</details>

---

**Next:** the single-cell resolution track in `../image/`, starting with `00_setup.ipynb`.